In [1]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.3/985.3 kB 11.7 MB/s eta 0:00:00


In [2]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 21.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
mlxtend 0.23.4 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
1.2.2
1.5.5


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [4]:
!pip install yfinance==0.2.54
from pandas_datareader import data as pdr
import yfinance as yf

In [5]:
def modify_date(date):
    return date[:10]

In [6]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [7]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [8]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [9]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [10]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [11]:
def output(weight):
    if stock_type == "S&P100":
      OEX=yf.download("^OEX", start=start_swan_date, end=end_swan_date)["Close"]

    elif stock_type == "S&P500":
      OEX=yf.download("SPY", start=start_swan_date, end=end_swan_date)["Close"]

    stock_price=yf.download(list(weight.index), start=start_swan_date, end=end_swan_date)["Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX.iloc[0,0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [12]:
# Preselection parameters set up
method = "K_means"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 19     #number of stock preselected
start_swan_date = dt(2020,2,15)
end_swan_date = dt(2021,3,23)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means
print('hello branch')

hello branch


In [13]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100_MVO2.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [14]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [15]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [16]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [17]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8
Swan1,1.0,-0.004268,0.111982,0.050073,0.001444,0.101963,0.079685,0.076826
Swan2,NaN,1.000000,0.549511,0.335485,0.410861,0.392490,0.033937,0.307858
Swan3,NaN,NaN,1.000000,0.151778,0.258652,0.435457,0.243705,0.224528
Swan4,NaN,NaN,NaN,1.000000,-0.032138,0.143125,0.147027,0.225411
Swan5,NaN,NaN,NaN,NaN,1.000000,0.444474,0.221512,0.208068
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.301317,0.482711
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.185835
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [18]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [19]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [20]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=True)
    else:
        preselected_stock = first_group

1.3617284057915178
Ticker
ACN     0
BKNG    0
BLK     3
GS      5
MDLZ    2
       ..
UNP     3
USB     3
WFC     3
WMT     4
XOM     1
Name: Group, Length: 86, dtype: int32


/usr/local/lib/python3.11/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
<ipython-input-20-7b7f6a0f143c>:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]


In [21]:
preselected_stock.to_csv('temp.csv')

In [22]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [23]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.head(selected_stock_number)

<ipython-input-23-dbdf54563bfa>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  preselected_stock["PR"]=1-preselected_stock["avg"]


,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg,PR
Ticker,,,,,,,,
MRK,0.617451,0.756485,0.674926,0.995745,-0.019569,4,0.465416,0.534584
SO,-0.138738,0.669547,-0.279810,0.055182,-0.091814,4,0.465416,0.534584
MO,0.818853,0.448863,-0.137637,0.264024,1.030036,4,0.465416,0.534584
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,4,0.465416,0.534584
LLY,0.746268,0.920577,0.280671,0.780342,0.061466,4,0.465416,0.534584
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,4,0.465416,0.534584
NEE,0.414534,0.503894,0.415367,0.192033,-0.026635,4,0.465416,0.534584
SBUX,0.289968,1.077849,0.586288,0.664628,-0.489110,4,0.465416,0.534584
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,4,0.465416,0.534584


# Step 2 - Download data and adjuste price based on Swan Beta

In [24]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [25]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]

In [26]:
#download data from yahoo finance
yahooDataFrame= yf.download(list(selected_stock_list), start=start, end=end)["Close"]


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  18 of 18 completed


In [27]:
yahooDataFrame

Ticker,ABT,AMGN,CL,COST,EXC,JNJ,KO,LLY,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,
2019-01-02,62.330685,158.586411,51.095375,186.144623,25.249969,107.296318,38.957539,105.114426,30.678118,60.248032,36.555988,91.920593,31.420738,77.998962,56.432415,33.987122,40.079819,28.292690
2019-01-03,59.389019,156.173248,50.853912,182.199203,25.278612,105.591339,38.716812,101.847900,30.510132,59.012630,36.465572,91.062637,30.541679,77.452072,53.984558,34.469097,40.222904,28.147203
2019-01-04,61.084072,161.511826,51.328217,187.490067,25.565088,107.363510,39.488815,104.913116,31.294037,60.790028,37.137157,92.929962,31.239113,79.032928,55.774395,34.756725,40.323067,28.323009
2019-01-07,61.998852,163.685272,51.638683,188.180939,25.610929,106.674812,38.974133,105.480408,30.348375,60.120514,37.083347,92.130882,31.406202,78.716736,55.774395,34.678993,40.580635,28.656431
2019-01-08,61.254456,165.784348,52.216450,189.590042,26.006260,109.152496,39.414112,106.450317,30.522579,60.566853,37.354568,93.014076,31.551508,79.007271,55.818260,35.666264,41.768291,28.856485
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,80.846695,193.655487,68.813469,288.671539,28.552507,131.141586,51.171322,136.146393,30.682306,70.134109,59.906769,126.156708,28.779501,111.503174,78.602837,55.631100,45.390793,35.641438
2020-02-11,81.643265,194.966858,67.710312,285.810547,29.161257,131.253845,51.034286,134.967499,30.263899,69.798401,59.997154,126.520439,28.992630,109.726852,78.584869,55.753017,44.222858,35.687824
2020-02-12,81.295334,193.297867,67.567970,289.895172,29.113979,130.476608,50.880127,134.275146,30.151005,68.160904,59.580498,126.520439,28.726223,109.099365,79.536766,55.777405,43.876247,35.826996


In [28]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
swan_to_predict=9
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [29]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [30]:
market_drop.to_csv("market_drop.csv")

In [31]:
market_drop

,0
0,0.0000
1,0.0037
2,0.0074
3,0.0111
4,0.0148
...,...
278,0.0000
279,0.0000
280,0.0000
281,0.0000


In [32]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [33]:
yahooDataFrame


Ticker,ABT,AMGN,CL,COST,EXC,JNJ,KO,LLY,MO,MRK,NEE,PEP,PFE,PG,SBUX,SO,VZ,WMT
Date,,,,,,,,,,,,,,,,,,
2019-01-02,62.330685,158.586411,51.095375,186.144623,25.249969,107.296318,38.957539,105.114426,30.678118,60.248032,36.555988,91.920593,31.420738,77.998962,56.432415,33.987122,40.079819,28.292690
2019-01-03,59.286749,155.904312,50.766340,181.885449,25.235081,105.409507,38.650140,101.672514,30.457592,58.911008,36.402777,90.905824,30.489085,77.318697,53.891595,34.409740,40.153639,28.098733
2019-01-04,60.873694,160.955566,51.151438,186.844336,25.477040,106.993742,39.352813,104.551788,31.186258,60.580662,37.009254,92.609904,31.131523,78.760733,55.582304,34.637020,40.184191,28.225463
2019-01-07,61.678559,162.839655,51.371912,187.208774,25.478620,106.123718,38.772788,104.935484,30.191592,59.809924,36.891770,91.654924,31.243954,78.310076,55.486258,34.499838,40.370991,28.508389
2019-01-08,60.832525,164.642399,51.856775,188.284117,25.827125,108.400637,39.142622,105.717071,30.312335,60.149659,37.097264,92.373381,31.334176,78.463057,55.433775,35.420589,41.480585,28.657717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,80.846695,193.655487,68.813469,288.671539,28.552507,131.141586,51.171322,136.146393,30.682306,70.134109,59.906769,126.156708,28.779501,111.503174,78.602837,55.631100,45.390793,35.641438
2020-02-11,81.643265,194.966858,67.710312,285.810547,29.161257,131.253845,51.034286,134.967499,30.263899,69.798401,59.997154,126.520439,28.992630,109.726852,78.584869,55.753017,44.222858,35.687824
2020-02-12,81.295334,193.297867,67.567970,289.895172,29.113979,130.476608,50.880127,134.275146,30.151005,68.160904,59.580498,126.520439,28.726223,109.099365,79.536766,55.777405,43.876247,35.826996


In [34]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['ABT', 'AMGN', 'CL', 'COST', 'EXC', 'JNJ', 'KO', 'LLY', 'MO',
       'MRK', 'NEE', 'PEP', 'PFE', 'PG', 'SBUX', 'SO', 'VZ', 'WMT'],
      dtype=object)

# Step3 - MVO

In [35]:
#Set the paramter for MVO process
Portfolio_Date=end

In [36]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [37]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [38]:
R_mv_one_year.index

Index(['ABT', 'AMGN', 'CL', 'COST', 'EXC', 'JNJ', 'KO', 'LLY', 'MO', 'MRK',
       'NEE', 'PEP', 'PFE', 'PG', 'SBUX', 'SO', 'VZ', 'WMT'],
      dtype='object', name='Ticker')

In [39]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [40]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [41]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [42]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [43]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [44]:
# Read OPT-data
optdata = pd.read_excel("optdata_MVO2.xlsx",index_col=0,sheet_name = 1)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [45]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [46]:
preselected_stock

,Swan4,Swan5,Swan6,Swan7,Swan8,Group,avg,PR,Rfwd_adjusted,std_f,SR,PR_SR
Ticker,,,,,,,,,,,,
ABT,0.702188,0.474696,0.311079,1.073150,0.431921,4,0.465416,0.534584,0.166575,0.181244,0.919065,0.512494
AMGN,0.795355,0.502892,0.109292,0.843475,0.659977,4,0.465416,0.534584,0.239498,0.244906,0.977918,0.528649
CL,0.099788,0.617542,-0.045657,0.784583,0.569564,4,0.465416,0.534584,0.156879,0.163610,0.958863,0.523473
COST,0.829818,0.976294,-0.015790,0.618966,0.918643,4,0.465416,0.534584,0.138570,0.153983,0.899904,0.507124
EXC,0.519667,0.794675,0.263389,0.605246,-0.123438,4,0.465416,0.534584,0.564181,0.156205,3.611792,1.015962
JNJ,0.403161,0.739963,0.360346,0.668742,0.555245,4,0.465416,0.534584,0.215852,0.155170,1.391069,0.630508
KO,-0.170562,0.536675,0.325968,0.614677,-0.049526,4,0.465416,0.534584,0.128161,0.168040,0.762680,0.466861
LLY,0.746268,0.920577,0.280671,0.780342,0.061466,4,0.465416,0.534584,0.315149,0.201647,1.562874,0.668310
MO,0.818853,0.448863,-0.137637,0.264024,1.030036,4,0.465416,0.534584,0.367903,0.222587,1.652848,0.687278


In [47]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [48]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [49]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


/usr/local/lib/python3.11/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


OrderedDict([('ABT', 0.0), ('AMGN', 0.01031), ('CL', 0.0), ('COST', 0.17), ('EXC', 0.0), ('JNJ', 0.0), ('KO', 0.06203), ('LLY', 0.12937), ('MO', 0.06724), ('MRK', 0.05052), ('NEE', 0.17), ('PEP', 0.0), ('PFE', 0.00054), ('PG', 0.0), ('SBUX', 0.0), ('SO', 0.17), ('VZ', 0.0), ('WMT', 0.17)])
Expected annual return: 11.4%
Annual volatility: 8.1%
Sharpe Ratio: 1.28


(0.11391499274585284, 0.08137349969252018, 1.2770127023970763)

In [50]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [51]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 1.2770020885360573


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [52]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.03]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [53]:
if stock_type == "S&P100":
    OEX=yf.download("^OEX", start=start_swan_date, end=end_swan_date)["Close"]

elif stock_type == "S&P500":
    OEX=yf.download("SPY", start=start_swan_date, end=end_swan_date)["Close"]

stock_price=yf.download(list(weight.index), start=start_swan_date, end=end_swan_date)["Close"]

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  8 of 8 completed


In [54]:
stock_price.to_csv("tradeinfo.csv")
stock_price

Ticker,COST,KO,LLY,MO,MRK,NEE,SO,WMT
Date,,,,,,,,
2020-02-18,296.270325,50.982906,132.619873,30.031464,67.514099,62.177483,56.918900,36.995960
2020-02-19,297.162689,51.188450,132.450562,29.978334,67.137466,60.552708,56.270977,36.392921
2020-02-20,298.137878,51.145626,133.550964,30.257265,67.538651,61.072994,57.255154,36.396008
2020-02-21,296.178436,51.496750,133.588562,30.476421,67.415840,61.196438,56.582638,36.671249
2020-02-24,288.515228,50.229248,129.930023,29.088411,66.588913,60.391769,55.606659,35.972336
...,...,...,...,...,...,...,...,...
2021-03-16,311.521912,45.744755,182.958176,35.887955,65.455025,68.198669,52.498726,41.939907
2021-03-17,313.368683,45.762623,178.961655,36.134766,65.872574,66.009216,52.027313,41.590904
2021-03-18,307.457184,45.164238,176.278107,36.708218,65.847015,64.353539,51.770172,41.047863


In [55]:
start=stock_price.loc["2020-02-19"]
end = stock_price.loc["2020-03-23"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)

In [56]:
stock_drop

,0
Ticker,
COST,0.116061
KO,0.366080
LLY,0.154654
MO,0.304829
MRK,0.183554
NEE,0.335171
SO,0.369917
WMT,0.024595


In [57]:
print(stock_price.head(1))
weight


Ticker            COST         KO         LLY         MO        MRK  \
Date                                                                  
2020-02-18  296.270325  50.982906  132.619873  30.031464  67.514099   

Ticker            NEE       SO       WMT  
Date                                      
2020-02-18  62.177483  56.9189  36.99596  


,weight
COST,0.171863
KO,0.062710
LLY,0.130788
MO,0.067977
MRK,0.051074
NEE,0.171863
SO,0.171863
WMT,0.171863


In [58]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [59]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX.iloc[0,0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")

In [60]:
temp=injected_result.loc[["2020-02-19","2020-03-23"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.30751759350367425


<ipython-input-60-0cde1dbf7bf2>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-60-0cde1dbf7bf2>:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2020-02-19,9909.495793,10051.749340
2020-03-23,7745.777304,6882.312479


In [61]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-10-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-10-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-10-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
NEE,0.171863,2.283184,18.241046,0.125,0.287547,0.17,0.17
WMT,0.171863,1.947176,13.267162,0.125,0.24523,0.17,0.17
SO,0.171863,0.724563,1.837045,0.125,0.091252,0.123382,0.17
MO,0.067977,0.687278,1.652848,0.125,0.086557,0.118686,0.17
LLY,0.130788,0.66831,1.562874,0.125,0.084168,0.116297,0.17
MRK,0.051074,0.655708,1.504489,0.125,0.082581,0.11471,0.05
COST,0.171863,0.507124,0.899904,0.125,0.063868,0.095997,0.05
KO,0.062710,0.466861,0.76268,0.125,0.058797,0.090927,0.05


In [62]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  8 of 8 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  8 of 8 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  8 of 8 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  8 of 8 completed


#